# Session 14 — Reproducibility, ethics and communication
*Séance 14 — Reproductibilité, éthique et diffusion*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noe2019/ML4EO_Bootcamp/blob/main/notebooks/14_diffusion_streamlit.ipynb)

**Course:** Machine Learning for Earth Observation (45 h) · **Instructor:** Noé Carème Fouotsa Manfouo

| Training track · Parcours de formation | This session · Cette séance |
|---|---|
| Geomatics / environment · Géomatique / environnement (UdeS géomatique appliquée; Bishop's Environmental Studies & Geography) | ● full session / séance complète |
| Computer science · Informatique (UdeS informatique; Bishop's Computer Science) | ● full session / séance complète |
| College / continuing education · Cégep / formation continue | ● full session / séance complète |

**By the end of this session you will be able to:**

1. Make an analysis reproducible (environment file, seeds, Git, project structure).
2. Assess the ethical risks of an EO/ML project with a checklist.
3. Save a trained model and serve it in an interactive Streamlit app.
4. Deploy the app from GitHub (Streamlit Community Cloud or Heroku).

**How to use this notebook:** read the lecture cells, run every code cell (`Shift + Enter`), then complete the cells marked
`# YOUR CODE HERE` in the lab section. Run the ✅ check cells to get instant feedback.

In [ ]:
# --- Setup (run once) -------------------------------------------------------
import sys, pathlib, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not pathlib.Path("ML4EO_Bootcamp").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/noe2019/ML4EO_Bootcamp.git"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *"geopandas rasterio rioxarray streamlit joblib".split()], check=True)
    ROOT = pathlib.Path("ML4EO_Bootcamp")
else:
    ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
DATA = ROOT / "data"
sys.path.append(str(ROOT / "notebooks"))   # course helper module (ml4eo_data.py)


def check(condition, message="Check failed"):
    """Instant feedback: raises an error with a hint if the condition is False."""
    assert condition, message
    print("✅ Passed")


def check_equal(value, expected, message=None):
    assert value == expected, message or f"Expected {expected!r}, got {value!r}"
    print("✅ Passed")

## Part A — Lecture: reproducibility

A result is **reproducible** when someone else can obtain it again from your materials. Minimum standard for the term project:

| Practice | How |
|---|---|
| Fixed environment | `environment.yml` or `requirements.txt` with versions (`pip freeze > requirements.txt`) |
| Fixed randomness | `random_state=`, `seed=` everywhere (sampling, splits, models) |
| Version control | Git + GitHub: commit early, meaningful messages, no data > 50 MB (use releases or Zenodo) |
| Clear structure | `data/` (raw, never modified), `notebooks/`, `src/`, `outputs/`, `README.md` |
| Runs end to end | *Restart & Run all* before submitting |

Essential Git commands: `git clone`, `git add`, `git commit -m "…"`, `git push`, `git pull`.

### Ethics checklist for EO/ML projects
1. **Privacy** — can individuals or homes be identified (very high resolution, population layers)?
2. **Representativeness** — were training data collected in the regions and seasons where the model is used?
3. **Error and its consequences** — who is affected by false positives / negatives (e.g. drought aid, flood maps)?
4. **Communication of uncertainty** — are accuracies and confidence intervals reported with the map?
5. **Environmental cost** — is a deep model necessary, or does a random forest do as well?
6. **Data licences and credit** — are sources cited and licences respected?

## Part B — Lab: from model to app

In [ ]:
import joblib, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from ml4eo_data import training_polygons, BANDS

df = training_polygons(seed=0)
FEATURES = BANDS + ["NDVI", "NDWI", "NDBI"]
model = RandomForestClassifier(300, random_state=0, n_jobs=-1).fit(df[FEATURES], df["label"])
joblib.dump({"model": model, "features": FEATURES}, "landcover_rf.joblib")
print("Saved model with classes:", list(model.classes_))

**Task 1 — Write the app.** Complete `APP` so that the Streamlit app: shows a title; asks for the reflectance of B3, B4, B8 and
B11 with `st.slider` (other bands = typical values provided); computes NDVI; predicts the class with the saved model;
displays the class and the class probabilities. Then write it to `app.py`.

In [ ]:
# YOUR CODE HERE
APP = """
import streamlit as st
# ...
"""
open("app.py", "w").write(APP)
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
from streamlit.testing.v1 import AppTest
import pathlib
at = AppTest.from_file(str(pathlib.Path("app.py").resolve())).run()
check(not at.exception, "The app raised an error")
check(len(at.slider) == 4, "Use four sliders")
print(at.subheader[0].value)

**Task 2 — Deployment files.** Write `requirements.txt` (streamlit, scikit-learn, pandas, joblib — with the versions installed
here) and, for Heroku, a `Procfile` containing
`web: streamlit run app.py --server.port=$PORT --server.address=0.0.0.0`.

In [ ]:
# YOUR CODE HERE

raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check(pathlib.Path("Procfile").read_text().startswith("web: streamlit run app.py"))
check("scikit-learn==" in pathlib.Path("requirements.txt").read_text())

### Deploy
1. Create a GitHub repository with `app.py`, `landcover_rf.joblib`, `requirements.txt` (and `Procfile` for Heroku).
2. **Streamlit Community Cloud** (free): <https://share.streamlit.io> → *New app* → pick the repository → *Deploy*.
3. **Heroku** (paid Eco dynos): create an app, connect the GitHub repository, *Deploy branch*.

Examples of deployed EO/ML apps by the instructor: [water storage forecasting](https://noe-water-storage-34fdf1488682.herokuapp.com/),
[invasive species mapping](https://noe-invasive-species-814a7fc4c770.herokuapp.com/),
[food security monitoring](https://noe-food-security-c984fe0b9737.herokuapp.com/).

**Run it in Colab (optional):** `!streamlit run app.py &>/content/log.txt &` then expose port 8501 with a tunnel service.

**❓ Question —** Apply the ethics checklist to your own term project: which two risks are the most serious, and how will you mitigate them?

*Your answer here.*

## Summary

- Reproducibility = fixed environment + seeds + version control + clear structure.
- An ethics checklist makes risks explicit before they reach users.
- A saved model + a 30-line Streamlit app turns an analysis into a tool others can use.

**To go further:**

- The Turing Way — reproducible research: https://book.the-turing-way.org
- Streamlit documentation: https://docs.streamlit.io